In [5]:
import os
import pandas as pd
import pdfplumber

In [4]:
def save_table_to_csv(df, output_dir, file_name):
    """Saves tables in CSV fomat in the file system."""
    os.makedirs(output_dir, exist_ok=True)
    full_path = os.path.join(output_dir, file_name)

    if os.path.exists(full_path):
        print(f"Skipping: {file_name} already exists in {output_dir}")
        return False # Signal that we didn't save anything

    df.to_csv(full_path, index=False)
    print(f"Successfully saved: {full_path}")
    return True

In [6]:
def extract_tables_from_pdf(file_path, output_dir, target_pages):
    """Main orchestrator for opening the PDF and processing pages."""
    with pdfplumber.open(file_path) as pdf:
        for page_index, file_name in target_pages.items():
            page = pdf.pages[page_index]
            table = page.extract_table()
            
            if not table:
                print(f"Warning: No table found on page {page_index + 1}")
                continue

            # Convert to DataFrame (skipping header row for data)
            df = pd.DataFrame(table[1:], columns=table[0])

            # Clean column names and data
            df.columns = [str(c).replace('\n', ' ') if c else f"Unnamed_{i}" 
                          for i, c in enumerate(df.columns)]
            df = df.replace('\n', ' ', regex=True)
            
            # Delegate the saving logic to the new function
            save_table_to_csv(df, output_dir, file_name)